# Linear predictors (reflex models)

_Artificial Intelligence (CS221)_

**Turn an input into numbers, take a dot product, read off a yes/no answer.**

A reflex model looks at an input and instantly gives an answer. No planning, no thinking ahead.

     First we turn the input into a list of numbers. That list is the feature vector.

---

This notebook is a practice scaffold for the **Linear predictors (reflex models)** lesson. We'll score emails for spam with a single dot product, then fit a real classifier on tumour scans and draw the line it learns. Run each cell, read the note above it, then try the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

A linear predictor scores an input `x` as `w · x` — a weighted sum of its features — and reads the **sign** of that score as the class. We'll classify four emails as spam (`+1`) or ham (`-1`) using hand-picked weights, in three steps: (1) set up features, labels and weights, (2) score and predict, (3) check how confident and correct each call was.

### Step 1 — Features, labels, and weights

Each email becomes a **feature vector**: the count of the word "free", the number of links, and a constant `1`. That trailing `1` is the **bias** feature — its weight shifts the decision threshold so the boundary needn't pass through the origin.

`w` holds one weight per feature. Positive weights push the score toward spam; the bias weight (`-3.0`) sets a baseline skepticism that the evidence must overcome.

In [ ]:
# Features per email: [count of "free", number of links, bias=1].
X = np.array([
    [2, 3, 1],
    [0, 1, 1],
    [4, 2, 1],
    [1, 0, 1],
])

y = np.array([+1, -1, +1, -1])      # true labels: +1 spam, -1 not spam (ham)
w = np.array([1.5, 0.5, -3.0])      # learned weights; last entry is the bias

print("X shape:", X.shape, " weights:", w)

### Step 2 — Score each email and predict

The matrix product `X @ w` computes every email's score `w · x` at once. Taking `np.sign` of those scores turns each real number into a hard `+1`/`-1` decision: a positive score predicts spam, a negative score predicts ham.

In [ ]:
scores = X @ w                      # one dot-product score per email
pred = np.sign(scores)              # +1 = spam, -1 = ham

print("scores     :", scores)
print("predictions:", pred.astype(int))

### Step 3 — Measure margin and accuracy

The **margin** is `score · true_label`. It is large and positive only when the model is both **correct** and **confident**; it goes negative whenever the prediction is wrong. Printing the margin per email shows not just *whether* the model is right but *how strongly*. Finally we compare predictions to the truth for an accuracy figure.

In [ ]:
margin = scores * y                 # >0 means correct; bigger means more confident

for i in range(len(y)):
    tag = "spam" if pred[i] > 0 else "ham "
    print("email", i,
          "score=%6.2f" % scores[i],
          "->", tag,
          "margin=%6.2f" % margin[i])

print("accuracy:", np.mean(pred == y))

## Visualize the data & results

_On the Wisconsin breast-cancer scan data, which tumors does a linear classifier call benign vs malignant?_

We swap the toy emails for real data and let scikit-learn *learn* the weights instead of guessing them. Same idea — a linear score and a sign — but now we can draw the **decision line** the model settles on.

### First, look at the data

Before we train on the breast-cancer scans, here's the full dataset — every feature column, the two class labels, and a few real rows. (The training code below uses only a couple of these columns so the result is easy to plot.)

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

### Step 1 — Load, scale, and fit a linear classifier

We keep two features (mean radius and mean texture) so the result is plottable in 2D, then **standardise** them so neither feature dominates the scale. `LogisticRegression` is a linear predictor: it learns a weight vector `w` and bias `b` and decides by the sign of `w · x + b`.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data[:, [0, 1]]            # mean radius, mean texture
y = data.target                    # 1 = benign, 0 = malignant

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

clf = LogisticRegression().fit(X_scaled, y)
print("train accuracy", clf.score(X_scaled, y))

### Step 2 — Convert the learned weights into a boundary line

The decision boundary is where the score is exactly zero: `w · z + b = 0`, in *standardised* coordinates `z`. Solving that line for the texture given a radius — and undoing the standardisation with the stored mean and scale — lets us draw the boundary back in the original, human-readable units.

In [ ]:
w, b = clf.coef_[0], clf.intercept_[0]
feat_mean, feat_scale = scaler.mean_, scaler.scale_

def boundary_texture(radius):
    # Standardise the radius, solve w0*z0 + w1*z1 + b = 0 for z1, then un-standardise.
    z0 = (radius - feat_mean[0]) / feat_scale[0]
    z1 = -(w[0] * z0 + b) / w[1]
    return z1 * feat_scale[1] + feat_mean[1]

### Step 3 — Plot the tumors and the decision line

Each point is one tumour: red for malignant, blue for benign. The orange line is the learned boundary — the model calls everything on one side benign and the other side malignant. The clouds overlap, which is exactly why no straight line separates them perfectly.

In [ ]:
malignant = y == 0
benign = y == 1

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X[malignant, 0], X[malignant, 1], c="#ff7b72", s=14, label="malignant")
ax.scatter(X[benign, 0], X[benign, 1], c="#4ea1ff", s=14, label="benign")

radii = np.linspace(10, 20, 20)
boundary = [boundary_texture(r) for r in radii]
ax.plot(radii, boundary, "#ffb454", lw=3, label="decision line")

ax.set_xlabel("mean radius")
ax.set_ylabel("mean texture")
ax.set_ylim(5, 40)
ax.legend()
ax.set_title("breast tumors and the learned decision line")
plt.show()